# 5 - Split temporal — treino, validação e teste

## Objetivo

Dividir o dataset normalizado em três conjuntos por **critério temporal**, refletindo o cenário real de uso do modelo: GAIN treinada em dados passados e aplicada em dados novos (anos posteriores).

## Posição na Etapa 2

Notebook **5 de 6** do pré-processamento. Consome `dataset_normalizado.parquet` (de `04_normalizacao.ipynb`). Entrega `train.parquet`, `val.parquet`, `test.parquet` + `split_info.json` para `06_mascaras.ipynb`.

## Decisão de split

| Split | Anos       | n esperado | Papel                                            |
|-------|------------|-----------:|--------------------------------------------------|
| Train | 2012–2022  | 533        | Treinar gerador/discriminador                    |
| Val   | 2023       |  34        | Hiperparâmetros, parada antecipada, diagnóstico  |
| Test  | 2024–2025  |  90        | Avaliação final em dados realmente novos         |

### Por que temporal, não stratified por estação?

- Stratified manteria a distribuição de cada estação em todos os splits, mas **vazaria informação do futuro para o passado** — o modelo teria visto coletas de 2024 enquanto era avaliado em 2014.
- O alvo real do projeto é **imputação prospectiva**: aplicar a GAIN em dados novos que chegam, não reconstruir o histórico.
- Temporal também testa **robustez ao layout**: 2024 e 2025 têm formato diferente (ver `Code/FetchAndTreatRawData/treat_raw_data.ipynb`) — o test set contém exatamente essas duas safras.

### Trade-off aceito

Algumas variáveis e estações **ficam ausentes** em val/test por terem sido descontinuadas ou só medidas em janelas específicas. Não invalida o protocolo da GAIN, que avalia via **máscara artificial** sobre células `M=1` — o sinal a aprender é a relação entre variáveis num registro, não a presença bruta delas em cada janela temporal.

### Gaps de cobertura propagados do `Code/2 - EDA/resumo.md`

- **Cianobactérias** medidas só em 2012 e 2016 → ausentes em val e test.
- **Microcistinas** medidas só em 2019 e 2023 → presentes em train (2019) e val (2023); ausentes em test.
- **Coliformes Termotolerantes** descontinuados em 2023 → ausentes em test.
- **Sólidos Suspensos Totais** retomados em 2023–2025 → cobertos nos três splits, mas concentrados nesses anos.
- **TJ306** sem coletas em 2024–2025 → ausente em test.
- **JC341** (n=10) e **MR363** (n=8) descontinuadas após 2015 → presentes apenas em train.
- **2020** com apenas 6 coletas (efeito COVID) — diluído em train; não invalida o split.


## Setup

Imports + caminhos relativos a `Code/3 - Preprocessing/05_split.ipynb`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

IN_PARQUET      = Path("../../Data/ProcessedData/dataset_normalizado.parquet")
IN_COLUMNS_JSON = Path("../../Data/ProcessedData/encoded_columns.json")

OUT_DIR    = Path("../../Data/GoldData/Splited")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_TRAIN  = OUT_DIR / "train.parquet"
OUT_VAL    = OUT_DIR / "val.parquet"
OUT_TEST   = OUT_DIR / "test.parquet"
OUT_INFO   = OUT_DIR / "split_info.json"

# Cortes temporais
ANO_TRAIN_FIM = 2022
ANO_VAL       = 2023
ANO_TEST_INI  = 2024

# Estações descontinuadas e variáveis com cobertura intermitente (esperado)
ESTACOES_DESCONTINUADAS_2015 = ["JC341", "MR363"]
ESTACAO_AUSENTE_2024_2025    = "TJ306"
VARS_ESPERADAS_AUSENTES_VAL  = ["Cianobacterias"]
VARS_ESPERADAS_AUSENTES_TEST = ["Cianobacterias", "Microcistinas", "Coliformes Termotolerantes"]

## Carregamento

Lê o dataset normalizado e o índice de colunas. Usa `encoded_columns.json` para saber **quais colunas são variáveis numéricas** (versus features temporais, one-hot etc.) — é o que vamos usar para medir cobertura por split.

In [2]:
df = pd.read_parquet(IN_PARQUET)
print(f"Shape de entrada: {df.shape}")

with open(IN_COLUMNS_JSON, "r", encoding="utf-8") as f:
    encoded_columns = json.load(f)

VARS         = encoded_columns["numericas"]
ESTACOES_OH  = encoded_columns["estacao_onehot"]

print(f"Variáveis numéricas: {len(VARS)}")
print(f"Estações one-hot: {len(ESTACOES_OH)}")
print(f"Anos cobertos: {sorted(df['Ano_int'].unique().tolist())}")
print(f"\nContagem por ano:")
df["Ano_int"].value_counts().sort_index()

Shape de entrada: (657, 46)
Variáveis numéricas: 13
Estações one-hot: 8
Anos cobertos: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Contagem por ano:


Ano_int
2012    54
2013    73
2014    71
2015    61
2016    99
2017    36
2018    31
2019    35
2020     6
2021    36
2022    31
2023    34
2024    30
2025    60
Name: count, dtype: int64

## Divisão temporal

Filtros simples por `Ano_int`. Mantemos todas as colunas (incluindo `Data` e `Ano_int` para auditoria; o vetor de entrada da GAIN será construído na Etapa 4 com base em `encoded_columns.json`).

In [3]:
df_train = df[df["Ano_int"] <= ANO_TRAIN_FIM].copy().reset_index(drop=True)
df_val   = df[df["Ano_int"] == ANO_VAL].copy().reset_index(drop=True)
df_test  = df[df["Ano_int"] >= ANO_TEST_INI].copy().reset_index(drop=True)

n_train, n_val, n_test = len(df_train), len(df_val), len(df_test)
print(f"train ({ANO_TRAIN_FIM}-): {n_train}")
print(f"val   ({ANO_VAL}):       {n_val}")
print(f"test  (>={ANO_TEST_INI}):  {n_test}")
print(f"soma = {n_train + n_val + n_test} (total esperado = {len(df)})")

assert n_train + n_val + n_test == len(df), "Soma dos splits diverge do total — algum registro caiu fora."
assert n_train > 0 and n_val > 0 and n_test > 0, 'Algum split está vazio.'


train (2022-): 533
val   (2023):       34
test  (>=2024):  90
soma = 657 (total esperado = 657)


## Sanidade 1 — soma e exclusividade

Nenhum ano deve aparecer em mais de um split.

In [4]:
anos_train = set(df_train["Ano_int"].unique().tolist())
anos_val   = set(df_val["Ano_int"].unique().tolist())
anos_test  = set(df_test["Ano_int"].unique().tolist())

assert anos_train.isdisjoint(anos_val),  f"Anos compartilhados train∩val: {anos_train & anos_val}"
assert anos_train.isdisjoint(anos_test), f"Anos compartilhados train∩test: {anos_train & anos_test}"
assert anos_val.isdisjoint(anos_test),   f"Anos compartilhados val∩test: {anos_val & anos_test}"

print(f"train: {sorted(anos_train)}")
print(f"val:   {sorted(anos_val)}")
print(f"test:  {sorted(anos_test)}")
print("\nSem sobreposição temporal — vazamento descartado.")

train: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
val:   [2023]
test:  [2024, 2025]

Sem sobreposição temporal — vazamento descartado.


## Cobertura por variável em cada split

Para cada uma das 13 variáveis numéricas, contar quantos registros têm valor válido (não-NaN) em cada split. Variáveis com `0` em algum split são as **esperadas** pelos gaps de cobertura registrados no `resumo.md` da EDA — destacamos explicitamente para confirmar que é decisão consciente, não bug.

In [5]:
def cobertura(df_sub):
    return {v: int(df_sub[v].notna().sum()) for v in VARS}

cov = pd.DataFrame({
    "train": cobertura(df_train),
    "val":   cobertura(df_val),
    "test":  cobertura(df_test),
})
cov["total"] = cov.sum(axis=1)
cov_display = cov.copy()
cov_display.index.name = "variavel"
print(cov_display)

# Variáveis com 0 em algum split — devem bater com a expectativa
vars_zero_val  = cov[cov["val"]  == 0].index.tolist()
vars_zero_test = cov[cov["test"] == 0].index.tolist()

print(f"\nVariáveis com 0 cobertura em val:  {vars_zero_val}")
print(f"Variáveis com 0 cobertura em test: {vars_zero_test}")

# Asserção que o que falta bate com o que a EDA já antecipou
assert set(vars_zero_val) <= set(VARS_ESPERADAS_AUSENTES_VAL), (
    f"Inesperado em val: {set(vars_zero_val) - set(VARS_ESPERADAS_AUSENTES_VAL)}"
)
assert set(vars_zero_test) <= set(VARS_ESPERADAS_AUSENTES_TEST), (
    f"Inesperado em test: {set(vars_zero_test) - set(VARS_ESPERADAS_AUSENTES_TEST)}"
)
print("\nGaps de cobertura batem com o esperado pela EDA — registro consciente, não bug.")

                            train  val  test  total
variavel                                           
DBO                           530   34    78    642
OD                            527   34    74    635
Nitrato                       252   12    73    337
Nitrogênio Amoniacal Total    386   34    77    497
Fósforo Total                 530   34    60    624
Condutividade                 531   32    72    635
pH                            526   34    79    639
Turbidez                      527   34    70    631
Temperatura da Água           519   34    76    629
Sólidos Suspensos Totais      126   15    16    157
Coliformes Termotolerantes    492    2     0    494
Cianobacterias                133    0     0    133
Microcistinas                  24   28     0     52

Variáveis com 0 cobertura em val:  ['Cianobacterias']
Variáveis com 0 cobertura em test: ['Coliformes Termotolerantes', 'Cianobacterias', 'Microcistinas']

Gaps de cobertura batem com o esperado pela EDA — registro cons

## Cobertura por estação em cada split

Soma das colunas `est_*` em cada split. Confirma:

- **JC341** e **MR363** aparecem apenas em train (descontinuadas após 2015).
- **TJ306** ausente em test (sem coletas em 2024–2025).
- As demais (CM320, JC342, MR361, MR369, TJ303) presentes nos três splits.

In [6]:
def contagem_estacoes(df_sub):
    return {col.replace("est_", ""): int(df_sub[col].sum()) for col in ESTACOES_OH}

cov_est = pd.DataFrame({
    "train": contagem_estacoes(df_train),
    "val":   contagem_estacoes(df_val),
    "test":  contagem_estacoes(df_test),
})
cov_est["total"] = cov_est.sum(axis=1)
cov_est.index.name = "estacao"
print(cov_est)

# Sanidade: estações descontinuadas em 2015 só em train
for est in ESTACOES_DESCONTINUADAS_2015:
    assert cov_est.loc[est, "val"] == 0,  f"{est} não deveria aparecer em val."
    assert cov_est.loc[est, "test"] == 0, f"{est} não deveria aparecer em test."

# TJ306 ausente em test
assert cov_est.loc[ESTACAO_AUSENTE_2024_2025, "test"] == 0, (
    f"{ESTACAO_AUSENTE_2024_2025} não deveria aparecer em test."
)
print("\nDescontinuadas (JC341, MR363) só em train: OK")
print(f"{ESTACAO_AUSENTE_2024_2025} ausente em test: OK")

         train  val  test  total
estacao                         
CM320       90    6    18    114
JC341       10    0     0     10
JC342       90    6    18    114
MR361       90    7    18    115
MR363        8    0     0      8
MR369       90    7    18    115
TJ303       89    6    18    113
TJ306       66    2     0     68

Descontinuadas (JC341, MR363) só em train: OK
TJ306 ausente em test: OK


## Sanidade 2 — features temporais sem extrapolação

`ano_norm` e `dias_desde_inicio` foram normalizadas em `04_normalizacao.ipynb` com base no min/max **global** (2012–2025). Como consequência:

- Val (2023) e test (2024–2025) **não extrapolam** o domínio normalizado visto em train (porque train já viu o ponto extremo `0` em 2012 e o ponto `+1` é 2025).
- Mas val/test ocupam apenas a **cauda direita** do range (`ano_norm` próximo de `+1`).

Confirmamos os ranges efetivos para que a Etapa 4 saiba que não há valores fora de `[-1, 1]` em nenhum split.

In [7]:
ranges = pd.DataFrame({
    split: {
        "ano_norm_min":          float(d["ano_norm"].min()),
        "ano_norm_max":          float(d["ano_norm"].max()),
        "dias_desde_inicio_min": float(d["dias_desde_inicio"].min()),
        "dias_desde_inicio_max": float(d["dias_desde_inicio"].max()),
    }
    for split, d in [("train", df_train), ("val", df_val), ("test", df_test)]
}).T
ranges.index.name = "split"
print(ranges)

# Todos os valores devem estar em [-1, 1] (foram normalizados em 04)
for split, d in [("train", df_train), ("val", df_val), ("test", df_test)]:
    for col in ["ano_norm", "dias_desde_inicio"]:
        assert d[col].between(-1 - 1e-9, 1 + 1e-9).all(), f"{col} fora de [-1, 1] em {split}"
print("\nTodas as features temporais escaladas dentro de [-1, 1] em todos os splits.")

       ano_norm_min  ano_norm_max  dias_desde_inicio_min  dias_desde_inicio_max
split                                                                          
train       -1.0000        0.5385                -1.0000                 0.5737
val          0.6923        0.6923                 0.6041                 0.7182
test         0.8462        1.0000                 0.7486                 1.0000

Todas as features temporais escaladas dentro de [-1, 1] em todos os splits.


## Persistência

Três parquets em `Data/GoldData/Splited/` + `split_info.json` com os metadados consumidos por `06_mascaras.ipynb` e pela Etapa 4.

In [8]:
df_train.to_parquet(OUT_TRAIN, index=False)
df_val.to_parquet(OUT_VAL,   index=False)
df_test.to_parquet(OUT_TEST, index=False)

split_info = {
    "n_train": n_train,
    "n_val":   n_val,
    "n_test":  n_test,
    "n_total": n_train + n_val + n_test,
    "anos_train": sorted(int(a) for a in anos_train),
    "anos_val":   sorted(int(a) for a in anos_val),
    "anos_test":  sorted(int(a) for a in anos_test),
    "cobertura_variaveis": {
        "train": cobertura(df_train),
        "val":   cobertura(df_val),
        "test":  cobertura(df_test),
    },
    "cobertura_estacoes": {
        "train": contagem_estacoes(df_train),
        "val":   contagem_estacoes(df_val),
        "test":  contagem_estacoes(df_test),
    },
    "gaps_conscientes": {
        "variaveis_ausentes_val":  vars_zero_val,
        "variaveis_ausentes_test": vars_zero_test,
        "estacoes_so_em_train":    ESTACOES_DESCONTINUADAS_2015,
        "estacao_ausente_test":    ESTACAO_AUSENTE_2024_2025,
    },
}
with open(OUT_INFO, "w", encoding="utf-8") as f:
    json.dump(split_info, f, ensure_ascii=False, indent=2)

print(f"Salvo: {OUT_TRAIN} ({OUT_TRAIN.stat().st_size} bytes)")
print(f"Salvo: {OUT_VAL}   ({OUT_VAL.stat().st_size} bytes)")
print(f"Salvo: {OUT_TEST}  ({OUT_TEST.stat().st_size} bytes)")
print(f"Salvo: {OUT_INFO}  ({OUT_INFO.stat().st_size} bytes)")

Salvo: ..\..\Data\GoldData\Splited\train.parquet (53123 bytes)
Salvo: ..\..\Data\GoldData\Splited\val.parquet   (31865 bytes)
Salvo: ..\..\Data\GoldData\Splited\test.parquet  (33420 bytes)
Salvo: ..\..\Data\GoldData\Splited\split_info.json  (2406 bytes)


## Síntese final

- **Split temporal sem vazamento**: train (2012–2022), val (2023), test (2024–2025). Soma fecha com o total do dataset normalizado.
- **Gaps conscientes registrados em `split_info.json`**:
  - **Cianobactérias** ausente em val e test (só medidas em 2012 e 2016).
  - **Microcistinas** e **Coliformes Termotolerantes** ausentes em test (descontinuadas/intermitentes).
  - **JC341** e **MR363** só aparecem em train (descontinuadas após 2015).
  - **TJ306** ausente em test (sem coletas em 2024–2025).
- **Features temporais** (`ano_norm`, `dias_desde_inicio`) permanecem em `[-1, 1]` em todos os splits — ranges globais foram fixados em `04_normalizacao.ipynb`.
- **Test set carrega o stress real** do projeto: anos com layout mudado (2024–2025) — testa robustez do pipeline ponta a ponta.

### Implicações para a Etapa 5 (Evaluation)

Variáveis ausentes em val/test (Cianobactérias, Microcistinas, Coliformes Termotolerantes) **não podem ser avaliadas via hold-out real** nesses splits. A avaliação dessas variáveis depende exclusivamente da **máscara artificial** sobre células `M=1` no treino — protocolo padrão da GAIN. Registro explícito para evitar discussões posteriores sobre "por que não temos métrica de Cianobactérias no test".

### Próximo notebook

`06_mascaras.ipynb` — define a máscara real `M` por split e a função geradora de máscara artificial `B` (taxa por variável: 0,2 robustas / 0,1 intermediárias / 0,05 críticas).